In [ ]:
# =============================================================================
# SUPERVISED LEARNING PIPELINE: CARER OUTCOME PREDICTION & INTERVENTION ANALYSIS
# =============================================================================
# This script provides an end-to-end supervised learning pipeline for
# Forward Carers assessment data. It covers:
#   1. Data loading and merging
#   2. Feature engineering (intervention flags from free text)
#   3. Binary outcome prediction (Low vs High scores per domain)
#   4. Train vs Test evaluation with confusion matrices and ROC/PR curves
#   5. Permutation importance (what drives outcomes)
#   6. Category-level effects (which groups are at risk)
#   7. Intervention impact modelling (which actions associate with improvement)
#   8. Intervention effect heatmap (cross-domain summary)
#
# Authors: [Team 6]
# Date: March 2026
# Hackathon: HealthTech AI Hub Hackathon 2025 - Forward Carers Challenge
# =============================================================================

In [4]:
#Importing Necessasary libraries
import re
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay
)

In [ ]:

# =============================================================================
# CONFIGURATION
# =============================================================================
# File paths - update these if your filenames differ
FEATURES_FILE = "features_master.csv"
SCORES_FILE = "survery_scores.csv"
REVIEW_FILE = "Hackathon_Forward_Carers_Review_Data.csv"
PROCESSED_FILE = "processed_data.csv"

# Output directory for all saved files
OUTPUT_DIR = "supervised_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Model settings
RANDOM_STATE = 0
N_SPLITS = 5
N_REPEATS_PERM = 10
IMPROVE_THRESHOLD = 1        # improved if review - baseline >= 1
REF_SAMPLE_N = 2000          # reference sample size for marginal effects
TOP_N_IMPORTANCE = 15        # number of features to show in importance plots
MIN_GROUP_SIZE = 50          # minimum group size for category effect plots

# Likert score labels
ALL_LABELS = np.array([1, 2, 3, 4, 5], dtype=int)

# Mapping from baseline score columns to review score columns
REVIEW_MAP = {
    "(11)  Your health and well being Score": "Your health and well being Score at review",
    "(16)  Work Education and Training Score": "Work Education and Training Score at review",
    "(19)  Your Financial Situation Score": "Your Financial Situation Score at review",
    "(23)  Time Out Score": "Time Out Score at review",
    "(26)  Other caring and family commitments Score": "Other caring and family commitments Score at review",
    "(30)  Relationships Score": "Relationships Score at review",
    "(34)  Your Home Score": "Your Home Score at review",
    "(37)  Your Diet Score": "Your Diet Score at review",
    "(40)  Your Safety Score": "How safe do you Feel score at review",
}

# Intervention keyword rules for extracting actions from free text
INTERVENTION_RULES = {
    "referral_cers": [
        r"\bCERS\b", r"Carers Emergency Response", r"emergency response service"
    ],
    "referral_social_services": [
        r"social services", r"adults? and communities", r"needs assessment"
    ],
    "acap_needs_assessment": [
        r"\bACAP\b", r"direct payment", r"\bDP\b", r"request a needs assessment"
    ],
    "ot_equipment": [
        r"occupational therapy", r"\bOT\b", r"equipment",
        r"bath[- ]?seat", r"walk[- ]?in shower", r"shower chair"
    ],
    "benefits_advice": [
        r"attendance allowance", r"\bAA\b", r"\bPIP\b", r"carers allowance",
        r"benefits", r"pension credit", r"housing benefit", r"council tax support"
    ],
    "support_groups_yoga": [
        r"support group", r"\byoga\b", r"buddhist",
        r"carers centre", r"mencap", r"jubilee"
    ],
    "respite_short_breaks": [
        r"short break", r"respite", r"sitting service", r"sitter", r"holiday"
    ],
    "wellbeing_payment": [
        r"well ?being payment", r"\b£\s*250\b", r"grant", r"payment of £"
    ],
    "signposting_info_only": [
        r"information (has been )?provided", r"leaflets",
        r"sent (by )?(post|email)", r"signposted"
    ],
    "no_actions": [
        r"no agreed actions", r"no agreed onward actions", r"no other help required"
    ],
}


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def safe_name(s):
    """Convert a string to a filesystem-safe name."""
    return "".join(ch if ch.isalnum() else "_" for ch in str(s))[:140]


def short_domain(s):
    """Shorten a domain column name for display (e.g. remove leading number)."""
    s = str(s)
    s = re.sub(r"^\(\d+\)\s*", "", s)
    s = s.replace("Score", "").strip()
    return s


def clean_interv_label(s):
    """Clean an intervention column name for display."""
    return s.replace("_", " ").title()


def parse_review_score(x):
    """Parse review score strings like '3 - Sometimes' to integer 3."""
    if pd.isna(x):
        return np.nan
    m = re.match(r"\s*([1-5])\s*", str(x))
    return float(m.group(1)) if m else np.nan


def binarise_likert(y):
    """
    Convert Likert 1-5 scores to binary:
      Low  = 1 or 2 -> 0
      High = 4 or 5 -> 1
      Score 3 is dropped (ambiguous middle)
    """
    y = pd.to_numeric(y, errors="coerce")
    yb = pd.Series(np.nan, index=y.index, dtype="float")
    yb[y.isin([1, 2])] = 0  # Low
    yb[y.isin([4, 5])] = 1  # High
    return yb


def make_flags(text_series):
    """
    Extract intervention flags from free-text summary actions.
    Uses keyword regex rules defined in INTERVENTION_RULES.
    Returns a DataFrame of 0/1 columns, one per intervention type.
    """
    txt = text_series.fillna("").astype(str)
    out = pd.DataFrame(index=txt.index)
    for name, patterns in INTERVENTION_RULES.items():
        regex = "(" + "|".join(patterns) + ")"
        out[name] = txt.str.contains(
            regex, flags=re.IGNORECASE, regex=True
        ).astype(int)
    return out


def make_hgb_pipeline(X_sample):
    """
    Build a preprocessing + HistGradientBoosting pipeline.
    Automatically detects categorical vs numeric columns.
    Uses regularised settings to reduce overfitting.
    """
    cat_cols = [c for c in X_sample.columns if X_sample[c].dtype == "object"]
    num_cols = [c for c in X_sample.columns if c not in cat_cols]

    preprocess = ColumnTransformer(
        transformers=[
            ("cat", Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
            ]), cat_cols),
            ("num", Pipeline([
                ("imp", SimpleImputer(strategy="median"))
            ]), num_cols),
        ],
        remainder="drop"
    )

    # Regularised HGB to reduce train-test gap
    model = HistGradientBoostingClassifier(
        max_depth=3,
        learning_rate=0.06,
        max_iter=600,
        min_samples_leaf=50,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.15
    )

    return Pipeline([("prep", preprocess), ("model", model)]), cat_cols, num_cols


def make_logreg_pipeline(X_sample):
    """
    Build a preprocessing + LogisticRegression pipeline.
    Uses balanced class weights for imbalanced targets.
    Good for interpretability (directional coefficients).
    """
    cat_cols = [c for c in X_sample.columns if X_sample[c].dtype == "object"]
    num_cols = [c for c in X_sample.columns if c not in cat_cols]

    preprocess = ColumnTransformer(
        transformers=[
            ("cat", Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
            ]), cat_cols),
            ("num", Pipeline([
                ("imp", SimpleImputer(strategy="median"))
            ]), num_cols),
        ],
        remainder="drop"
    )

    model = LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        class_weight="balanced",
        random_state=RANDOM_STATE
    )

    return Pipeline([("prep", preprocess), ("model", model)])


def pick_threshold(y_true, proba, objective="balanced_accuracy"):
    """
    Search for the probability threshold that maximises the chosen objective.
    Searches a grid from 0.05 to 0.95.
    Returns (best_threshold, best_score).
    """
    grid = np.linspace(0.05, 0.95, 91)
    best_t, best_s = 0.5, -np.inf
    for t in grid:
        pred = (proba >= t).astype(int)
        if objective == "balanced_accuracy":
            s = balanced_accuracy_score(y_true, pred)
        elif objective == "f1":
            s = f1_score(y_true, pred)
        else:
            raise ValueError("objective must be 'balanced_accuracy' or 'f1'")
        if s > best_s:
            best_s, best_t = s, t
    return best_t, best_s


# =============================================================================
# STEP 1: LOAD AND MERGE DATA
# =============================================================================
print("=" * 70)
print("STEP 1: Loading and merging data")
print("=" * 70)

# Load baseline features (demographics, caring intensity, conditions, tasks)
features = pd.read_csv(FEATURES_FILE)
print(f"  Baseline features: {len(features)} rows, {len(features.columns)} columns")

# Load baseline Likert scores (1-5 per domain)
baseline_scores = pd.read_csv(SCORES_FILE)
print(f"  Baseline scores: {len(baseline_scores)} rows")

# Load 6-month review data
review = pd.read_csv(REVIEW_FILE)
review = review.rename(columns=lambda c: c.strip() if isinstance(c, str) else c)
print(f"  Review data: {len(review)} rows")

# Load processed data (contains summary_actions free text)
proc = pd.read_csv(PROCESSED_FILE)
proc = proc.rename(columns=lambda c: c.strip() if isinstance(c, str) else c)
print(f"  Processed data: {len(proc)} rows")
print(f"  summary_actions coverage: {proc['summary_actions'].notna().mean():.1%}")

# Identify feature and target columns
feature_cols = [c for c in features.columns if c != "ID"]
target_cols = [c for c in baseline_scores.columns if c != "ID"]

print(f"\n  Feature columns ({len(feature_cols)}): {feature_cols[:5]}...")
print(f"  Target columns ({len(target_cols)}): {[short_domain(t) for t in target_cols]}")


# =============================================================================
# STEP 2: EXTRACT INTERVENTION FLAGS FROM FREE TEXT
# =============================================================================
print("\n" + "=" * 70)
print("STEP 2: Extracting intervention flags from summary_actions")
print("=" * 70)

# Extract 0/1 flags for each intervention type using keyword rules
actions = proc[["ID", "summary_actions"]].copy()
flags = make_flags(actions["summary_actions"])
actions = pd.concat([actions[["ID"]], flags], axis=1)
intervention_cols = list(INTERVENTION_RULES.keys())

# Print intervention prevalence
print("\n  Intervention prevalence (full baseline dataset):")
for ic in intervention_cols:
    rate = actions[ic].mean()
    count = actions[ic].sum()
    print(f"    {ic:30s}: {rate:6.1%}  (n={count})")


# =============================================================================
# STEP 3: BINARY OUTCOME PREDICTION (Low vs High per domain)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 3: Binary outcome prediction (Low=1-2 vs High=4-5)")
print("=" * 70)

# Merge features + scores for the full baseline dataset
df_baseline = features.merge(baseline_scores, on="ID", how="inner")
print(f"  Merged baseline rows: {len(df_baseline)}")

binary_summary_rows = []

for target in target_cols:
    # Binarise: Low (1-2) = 0, High (4-5) = 1, drop 3
    yb = binarise_likert(df_baseline[target])
    mask = yb.notna()
    X = df_baseline.loc[mask, feature_cols].copy()
    y = yb.loc[mask].astype(int)

    if y.nunique() < 2:
        print(f"  {short_domain(target)}: only one class after binarising; skipping.")
        continue

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    # Storage for fold metrics
    train_auc, test_auc = [], []
    train_bacc, test_bacc = [], []

    # Out-of-fold predictions for plots
    proba_oof = np.zeros(len(y), dtype=float)
    pred_oof = np.zeros(len(y), dtype=int)

    # Permutation importance across folds
    fold_imps = []

    for tr, te in cv.split(X, y):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]

        # Build and train pipeline
        pipe, cat_cols, num_cols = make_hgb_pipeline(Xtr)
        pipe.fit(Xtr, ytr)

        # Train predictions
        p_tr = pipe.predict_proba(Xtr)[:, 1]
        train_auc.append(roc_auc_score(ytr, p_tr))
        train_bacc.append(balanced_accuracy_score(ytr, pipe.predict(Xtr)))

        # Test predictions
        p_te = pipe.predict_proba(Xte)[:, 1]
        proba_oof[te] = p_te

        # Threshold tuning on TRAIN fold, apply to TEST fold
        t_star, _ = pick_threshold(ytr, p_tr, objective="balanced_accuracy")
        pred_oof[te] = (p_te >= t_star).astype(int)

        test_auc.append(roc_auc_score(yte, p_te))
        test_bacc.append(balanced_accuracy_score(yte, pred_oof[te]))

        # Permutation importance on test fold
        pi = permutation_importance(
            pipe, Xte, yte,
            n_repeats=N_REPEATS_PERM,
            random_state=RANDOM_STATE,
            scoring="balanced_accuracy"
        )
        fold_imps.append(pi.importances_mean)

    # Aggregate importance
    imp_mean = np.mean(np.vstack(fold_imps), axis=0)
    imp_std = np.std(np.vstack(fold_imps), axis=0)
    imp_df = pd.DataFrame({
        "feature": feature_cols,
        "perm_importance_mean": imp_mean,
        "perm_importance_std": imp_std
    }).sort_values("perm_importance_mean", ascending=False)

    # Save importance CSV
    dom_safe = safe_name(target)
    imp_df.to_csv(os.path.join(OUTPUT_DIR, f"binary_perm_importance_{dom_safe}.csv"), index=False)

    # OOF ROC-AUC and PR-AUC
    oof_auc = roc_auc_score(y, proba_oof)
    oof_ap = average_precision_score(y, proba_oof)

    # --- ROC curve plot ---
    fpr, tpr, _ = roc_curve(y.to_numpy(), proba_oof)
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"OOF AUC = {oof_auc:.3f}")
    plt.plot([0, 1], [0, 1], "k--", lw=1)
    plt.title(f"ROC (OOF) — {short_domain(target)}\nBinary: Low(1-2) vs High(4-5)")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"roc_{dom_safe}.png"), dpi=200)
    plt.close()

    # --- PR curve plot ---
    prec, rec, _ = precision_recall_curve(y.to_numpy(), proba_oof)
    plt.figure(figsize=(6, 5))
    plt.plot(rec, prec, label=f"OOF AP = {oof_ap:.3f}")
    plt.title(f"Precision–Recall (OOF) — {short_domain(target)}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"pr_{dom_safe}.png"), dpi=200)
    plt.close()

    # --- Confusion matrix plot ---
    cm = confusion_matrix(y, pred_oof, labels=[0, 1])
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ConfusionMatrixDisplay(cm, display_labels=["Low(1-2)", "High(4-5)"]).plot(
        ax=ax[0], cmap="Blues", values_format="d", colorbar=False
    )
    ax[0].set_title("OOF Confusion (counts)")
    ConfusionMatrixDisplay.from_predictions(
        y, pred_oof, labels=[0, 1], display_labels=["Low(1-2)", "High(4-5)"],
        normalize="true", cmap="Blues", values_format=".2f", ax=ax[1], colorbar=False
    )
    ax[1].set_title("OOF Confusion (normalized)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"confmat_{dom_safe}.png"), dpi=200)
    plt.close()

    # --- Permutation importance bar plot ---
    top = imp_df.head(TOP_N_IMPORTANCE).iloc[::-1]
    plt.figure(figsize=(8, 5))
    plt.barh(top["feature"], top["perm_importance_mean"])
    plt.xlabel("Permutation Importance (Δ Balanced Accuracy)")
    plt.title(f"Top {TOP_N_IMPORTANCE} Predictors — {short_domain(target)}")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"importance_{dom_safe}.png"), dpi=200)
    plt.close()

    # Store summary row
    binary_summary_rows.append({
        "domain": target,
        "domain_short": short_domain(target),
        "n_used": int(len(y)),
        "low_count": int((y == 0).sum()),
        "high_count": int((y == 1).sum()),
        "pct_high": float((y == 1).mean() * 100),
        "train_auc_mean": float(np.mean(train_auc)),
        "test_auc_mean": float(np.mean(test_auc)),
        "train_bacc_mean": float(np.mean(train_bacc)),
        "test_bacc_mean": float(np.mean(test_bacc)),
        "oof_auc": oof_auc,
        "oof_ap": oof_ap,
    })

    print(f"  {short_domain(target):40s} | "
          f"Train AUC {np.mean(train_auc):.3f} | "
          f"Test AUC {np.mean(test_auc):.3f} | "
          f"Test BAcc {np.mean(test_bacc):.3f}")

# Save binary prediction summary
binary_summary = pd.DataFrame(binary_summary_rows).sort_values("test_auc_mean", ascending=False)
binary_summary.to_csv(os.path.join(OUTPUT_DIR, "binary_prediction_summary.csv"), index=False)
print(f"\n  Saved: {OUTPUT_DIR}/binary_prediction_summary.csv")


# =============================================================================
# STEP 4: CATEGORY-LEVEL EFFECTS (which groups drive high/low outcomes)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 4: Category-level effects on P(High)")
print("=" * 70)

for target in target_cols:
    yb = binarise_likert(df_baseline[target])
    mask = yb.notna()
    X = df_baseline.loc[mask, feature_cols].copy()
    y = yb.loc[mask].astype(int)

    if y.nunique() < 2:
        continue

    # Fit on full data for interpretation
    pipe, cat_cols, num_cols = make_hgb_pipeline(X)
    pipe.fit(X, y)

    # Reference sample for marginal effects
    X_ref = X.sample(n=min(REF_SAMPLE_N, len(X)), random_state=RANDOM_STATE).copy()
    baseline_p = float(np.mean(pipe.predict_proba(X_ref)[:, 1]))

    # Compute effects for each categorical feature
    effects_all = []
    for c in cat_cols:
        levels = (
            X_ref[c].fillna("MISSING").astype(str).value_counts().head(30).index.tolist()
        )
        for lvl in levels:
            X_tmp = X_ref.copy()
            X_tmp[c] = lvl
            p_high = float(np.mean(pipe.predict_proba(X_tmp)[:, 1]))
            count = int((X_ref[c].fillna("MISSING").astype(str) == lvl).sum())
            effects_all.append({
                "feature": c,
                "level": lvl,
                "avg_p_high": p_high,
                "delta_vs_baseline": p_high - baseline_p,
                "baseline_avg_p_high": baseline_p,
                "level_count_in_ref": count
            })

    effects_df = pd.DataFrame(effects_all)
    dom_safe = safe_name(target)
    effects_df.to_csv(
        os.path.join(OUTPUT_DIR, f"category_effects_{dom_safe}.csv"), index=False
    )

    print(f"  {short_domain(target):40s} | {len(effects_df)} level effects saved")


# =============================================================================
# STEP 5: INTERVENTION IMPACT MODELLING (6-month review subset)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 5: Intervention impact modelling (review subset)")
print("=" * 70)

# Parse review scores to numeric
REVIEW_MAP_CLEAN = {k: v.strip() for k, v in REVIEW_MAP.items()}
for base_col, rev_col in REVIEW_MAP_CLEAN.items():
    if rev_col not in review.columns:
        print(f"  WARNING: Review column not found: '{rev_col}'")
        continue
    review[rev_col + "_num"] = review[rev_col].map(parse_review_score)

# Merge all data for the review subset
review_small = review[
    ["Assessment ID"] + [v + "_num" for v in REVIEW_MAP_CLEAN.values()]
].copy()
review_small = review_small.rename(columns={"Assessment ID": "ID"})

df_review = (
    features.merge(baseline_scores, on="ID", how="inner").merge(actions, on="ID", how="left").merge(review_small, on="ID", how="inner")
)
print(f"  Review subset after merge: {len(df_review)} rows")

# Run improvement models per domain
improvement_rows = []

for base_col, rev_col in REVIEW_MAP_CLEAN.items():
    rev_num_col = rev_col + "_num"

    # Compute baseline and review scores
    y0 = pd.to_numeric(df_review[base_col], errors="coerce").where(lambda s: s.isin([1, 2, 3, 4, 5]))
    y1 = pd.to_numeric(df_review[rev_num_col], errors="coerce").where(lambda s: s.isin([1, 2, 3, 4, 5]))
    valid = y0.notna() & y1.notna()

    d = df_review.loc[valid].copy()
    d["delta"] = y1.loc[valid].to_numpy() - y0.loc[valid].to_numpy()
    d["improved"] = (d["delta"] >= IMPROVE_THRESHOLD).astype(int)

    # Features: baseline features + baseline domain score + intervention flags
    X_cols = feature_cols + [base_col] + intervention_cols
    X = d[X_cols].copy()
    y = d["improved"].copy()

    if y.nunique() < 2:
        continue

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    # Collect metrics
    train_auc, test_auc = [], []
    train_bacc, test_bacc = [], []
    proba_oof = np.zeros(len(y), dtype=float)
    pred_oof = np.zeros(len(y), dtype=int)
    fold_imps = []

    for tr, te in cv.split(X, y):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]

        pipe = make_logreg_pipeline(Xtr)
        pipe.fit(Xtr, ytr)

        # Train metrics
        p_tr = pipe.predict_proba(Xtr)[:, 1]
        train_auc.append(roc_auc_score(ytr, p_tr))
        t_star, _ = pick_threshold(ytr, p_tr, objective="balanced_accuracy")
        train_bacc.append(balanced_accuracy_score(ytr, (p_tr >= t_star).astype(int)))

        # Test metrics
        p_te = pipe.predict_proba(Xte)[:, 1]
        proba_oof[te] = p_te
        pred_oof[te] = (p_te >= t_star).astype(int)

        test_auc.append(roc_auc_score(yte, p_te))
        test_bacc.append(balanced_accuracy_score(yte, pred_oof[te]))

        # Permutation importance
        pi = permutation_importance(
            pipe, Xte, yte,
            n_repeats=N_REPEATS_PERM,
            random_state=RANDOM_STATE,
            scoring="roc_auc"
        )
        fold_imps.append(pi.importances_mean)

    # Save importance
    imp_mean = np.mean(np.vstack(fold_imps), axis=0)
    imp_std = np.std(np.vstack(fold_imps), axis=0)
    imp_df = pd.DataFrame({
        "feature": X_cols,
        "perm_importance_mean_auc": imp_mean,
        "perm_importance_std": imp_std
    }).sort_values("perm_importance_mean_auc", ascending=False)

    dom_safe = safe_name(base_col)
    imp_df.to_csv(os.path.join(OUTPUT_DIR, f"improve_importance_{dom_safe}.csv"), index=False)

    # Intervention-only importance
    imp_interv = (
        imp_df[imp_df["feature"].isin(intervention_cols)].copy().sort_values("perm_importance_mean_auc", ascending=False)
    )
    imp_interv.to_csv(os.path.join(OUTPUT_DIR, f"improve_interv_importance_{dom_safe}.csv"), index=False)

    # OOF metrics
    oof_auc = roc_auc_score(y, proba_oof)
    oof_ap = average_precision_score(y, proba_oof)

    improvement_rows.append({
        "domain": base_col,
        "domain_short": short_domain(base_col),
        "n_review": int(len(y)),
        "improved_rate": float(y.mean()),
        "train_auc_mean": float(np.mean(train_auc)),
        "test_auc_mean": float(np.mean(test_auc)),
        "train_bacc_mean": float(np.mean(train_bacc)),
        "test_bacc_mean": float(np.mean(test_bacc)),
        "oof_auc": oof_auc,
        "oof_ap": oof_ap,
    })

    print(f"  {short_domain(base_col):40s} | "
          f"improved={y.mean():.1%} | "
          f"Train AUC {np.mean(train_auc):.3f} | "
          f"Test AUC {np.mean(test_auc):.3f}")

# Save improvement summary
improve_summary = pd.DataFrame(improvement_rows).sort_values("test_auc_mean", ascending=False)
improve_summary.to_csv(os.path.join(OUTPUT_DIR, "improvement_model_summary.csv"), index=False)
print(f"\n  Saved: {OUTPUT_DIR}/improvement_model_summary.csv")


# =============================================================================
# STEP 6: INTERVENTION EFFECT HEATMAP (cross-domain summary)
# =============================================================================
print("\n" + "=" * 70)
print("STEP 6: Intervention effect heatmap")
print("=" * 70)

# Build effect matrix: for each domain x intervention, compute delta P(improved)
effects = pd.DataFrame(
    index=[short_domain(k) for k in REVIEW_MAP_CLEAN.keys()],
    columns=intervention_cols, dtype=float
)

interv_prevalence = {}

for base_col, rev_col in REVIEW_MAP_CLEAN.items():
    rev_num_col = rev_col + "_num"

    y0 = pd.to_numeric(df_review[base_col], errors="coerce").where(lambda s: s.isin([1, 2, 3, 4, 5]))
    y1 = pd.to_numeric(df_review[rev_num_col], errors="coerce").where(lambda s: s.isin([1, 2, 3, 4, 5]))
    valid = y0.notna() & y1.notna()

    d = df_review.loc[valid].copy()
    d["delta"] = y1.loc[valid].to_numpy() - y0.loc[valid].to_numpy()
    d["improved"] = (d["delta"] >= IMPROVE_THRESHOLD).astype(int)

    X_cols = feature_cols + [base_col] + intervention_cols
    X = d[X_cols].copy()
    y_imp = d["improved"].copy()

    if y_imp.nunique() < 2:
        continue

    # Record prevalence once
    if not interv_prevalence:
        for ic in intervention_cols:
            interv_prevalence[ic] = float(X[ic].mean())

    # Fit logistic regression on full review subset for this domain
    pipe = make_logreg_pipeline(X)
    pipe.fit(X, y_imp)

    # Reference sample for marginal effects
    X_ref = X.sample(n=min(REF_SAMPLE_N, len(X)), random_state=RANDOM_STATE).copy()

    # Toggle each intervention ON vs

STEP 1: Loading and merging data
  Baseline features: 11769 rows, 51 columns
  Baseline scores: 11769 rows
  Review data: 2380 rows


/var/folders/89/hh10qt7x6kl9g2p1m6mx52r00000gn/T/ipykernel_37880/848705260.py:249: DtypeWarning: Columns (50) have mixed types. Specify dtype option on import or set low_memory=False.
  proc = pd.read_csv(PROCESSED_FILE)
/var/folders/89/hh10qt7x6kl9g2p1m6mx52r00000gn/T/ipykernel_37880/848705260.py:132: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  out[name] = txt.str.contains(
/var/folders/89/hh10qt7x6kl9g2p1m6mx52r00000gn/T/ipykernel_37880/848705260.py:132: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  out[name] = txt.str.contains(


  Processed data: 11767 rows
  summary_actions coverage: 99.3%

  Feature columns (50): ['Carer Age Bin', 'Sexual Orientation', 'Primary Cared For Age Bin', 'Primary Cared For Relationship', 'Primary Cared For Gender']...
  Target columns (9): ['Your health and well being', 'Work Education and Training', 'Your Financial Situation', 'Time Out', 'Other caring and family commitments', 'Relationships', 'Your Home', 'Your Diet', 'Your Safety']

STEP 2: Extracting intervention flags from summary_actions


/var/folders/89/hh10qt7x6kl9g2p1m6mx52r00000gn/T/ipykernel_37880/848705260.py:132: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  out[name] = txt.str.contains(
/var/folders/89/hh10qt7x6kl9g2p1m6mx52r00000gn/T/ipykernel_37880/848705260.py:132: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  out[name] = txt.str.contains(
/var/folders/89/hh10qt7x6kl9g2p1m6mx52r00000gn/T/ipykernel_37880/848705260.py:132: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  out[name] = txt.str.contains(
/var/folders/89/hh10qt7x6kl9g2p1m6mx52r00000gn/T/ipykernel_37880/848705260.py:132: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  out[name] = txt.str.contains(
/var/fol


  Intervention prevalence (full baseline dataset):
    referral_cers                 :  38.5%  (n=4527)
    referral_social_services      :  11.4%  (n=1346)
    acap_needs_assessment         :   7.1%  (n=836)
    ot_equipment                  :   5.3%  (n=626)
    benefits_advice               :  19.6%  (n=2306)
    support_groups_yoga           :  39.8%  (n=4679)
    respite_short_breaks          :  16.7%  (n=1964)
    wellbeing_payment             :  33.8%  (n=3972)
    signposting_info_only         :   6.7%  (n=783)
    no_actions                    :   0.2%  (n=23)

STEP 3: Binary outcome prediction (Low=1-2 vs High=4-5)
  Merged baseline rows: 11769
  Your health and well being               | Train AUC 0.772 | Test AUC 0.689 | Test BAcc 0.633
  Work Education and Training              | Train AUC 0.773 | Test AUC 0.743 | Test BAcc 0.682
  Your Financial Situation                 | Train AUC 0.825 | Test AUC 0.795 | Test BAcc 0.725
  Time Out                                 | Tra